Baseline Model

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score
)

from sklearn.ensemble import RandomForestClassifier
import matplotlib.pyplot as plt
from imblearn.over_sampling import SMOTE

In [ ]:
df = pd.read_csv("../data/raw/creditcard.csv")

X = df.drop("Class", axis=1)
y = df["Class"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

scaler_base = StandardScaler()

X_train_base_scaled = scaler_base.fit_transform(X_train)
X_test_base_scaled = scaler_base.transform(X_test)

In [4]:
baseline_model = LogisticRegression(
    class_weight="balanced",
    max_iter=1000,
    random_state=42
)

baseline_model.fit(X_train_base_scaled, y_train)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",'balanced'
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",42
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:

In [5]:
y_prob_base = baseline_model.predict_proba(X_test_base_scaled)[:,1]
y_pred_base = (y_prob_base >= 0.9).astype(int)

Randomforest

In [ ]:
rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    min_samples_split=10,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=1,
    class_weight="balanced_subsample"
)

rf_model.fit(X_train, y_train)

y_prob_rf = rf_model.predict_proba(X_test)[:,1]

threshold = 0.7
y_pred_rf = (y_prob_rf >= threshold).astype(int)

In [10]:
print(classification_report(y_test, y_pred_rf))
print(confusion_matrix(y_test, y_pred_rf))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00     56864
           1       0.77      0.84      0.80        98

    accuracy                           1.00     56962
   macro avg       0.88      0.92      0.90     56962
weighted avg       1.00      1.00      1.00     56962

[[56839    25]
 [   16    82]]


In [12]:
comparison = pd.DataFrame([
    {
        "model" : "Baseline",
        "precision" : precision_score(y_test, y_pred_base, zero_division=0),
        "recall" : recall_score(y_test, y_pred_base, zero_division=0),
        "f1" : f1_score(y_test, y_pred_base, zero_division=0),
        "roc-auc" : roc_auc_score(y_test, y_prob_base),
        "false_positives" : confusion_matrix(y_test, y_pred_base)[0,1],
        "false_negatives" : confusion_matrix(y_test, y_pred_base)[1,0],
        "true_positives" : confusion_matrix(y_test, y_pred_base)[1,1]

    },
    {
        "model" : "Feature Engineered",
        "precision" : precision_score(y_test, y_pred_rf, zero_division=0),
        "recall" : recall_score(y_test, y_pred_rf, zero_division=0),
        "f1" : f1_score(y_test, y_pred_rf, zero_division=0),
        "roc-auc" : roc_auc_score(y_test, y_prob_rf),
        "false_positives" : confusion_matrix(y_test, y_pred_rf)[0,1],
        "false_negatives" : confusion_matrix(y_test, y_pred_rf)[1,0],
        "true_positives" : confusion_matrix(y_test, y_pred_rf)[1,1]
    }])

comparison

,model,precision,recall,f1,roc-auc,false_positives,false_negatives,true_positives
0,Baseline,0.247159,0.887755,0.386667,0.972083,265,11,87
1,Feature Engineered,0.852273,0.765306,0.806452,0.982668,13,23,75


In [16]:
results = X_test.copy()

results["prob"] = y_prob_rf
results["score"] = (y_prob_rf*100).round().astype(int)
results["Class"] = df["Class"]
results["pred"] = y_pred_rf

top_risky = results.sort_values("score", ascending=False).head(10)
top_risky

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V24,V25,V26,V27,V28,Amount,prob,score,Class,pred
42009,40919.0,-2.740483,3.658095,-4.110636,5.340242,-2.666775,-0.092782,-4.388699,-0.280133,-2.821895,...,-0.154757,-0.403956,0.277895,0.830062,0.218690,112.33,0.999743,100,1,1
15476,26863.0,-21.209120,12.652197,-23.553933,6.174078,-16.026658,-4.422195,-16.229444,14.116002,-3.922741,...,0.130166,1.454857,-0.223214,1.550928,0.461460,99.99,0.999677,100,1,1
229712,146022.0,0.908637,2.849024,-5.647343,6.009415,0.216656,-2.397014,-1.819308,0.338527,-2.819883,...,-0.168597,0.465058,0.210510,0.648705,0.360224,1.18,0.999692,100,1,1
143333,85285.0,-7.030308,3.421991,-9.525072,5.270891,-4.024630,-2.865682,-6.989195,3.791551,-4.622730,...,-0.355519,0.353634,1.042458,1.359516,-0.272188,0.00,0.999748,100,1,1
77348,57007.0,-1.271244,2.462675,-2.851395,2.324480,-1.372245,-0.948196,-3.065234,1.166927,-2.268771,...,-0.523582,0.224228,0.756335,0.632800,0.250187,0.01,0.999665,100,1,1
6529,7891.0,-1.585505,3.261585,-4.137422,2.357096,-1.405043,-1.879437,-3.513687,1.515607,-1.207166,...,-0.425550,0.123644,0.321985,0.264028,0.132817,1.00,0.999317,100,1,1
151519,95628.0,-17.518909,12.572118,-19.038538,11.190895,-13.554721,-0.411924,-23.189397,-5.301412,-8.630390,...,0.334418,-0.720128,-0.232603,-3.021992,-0.478158,1.63,0.999587,100,1,1
143336,85285.0,-6.713407,3.921104,-9.746678,5.148263,-5.151563,-2.099389,-5.937767,3.578780,-4.684952,...,-0.339450,0.394096,1.075295,1.649906,-0.394905,252.92,0.999754,100,1,1
42696,41203.0,-8.426814,6.241659,-9.946470,8.199614,-8.213093,-2.522046,-11.643028,5.339500,-7.051016,...,0.499809,0.467594,0.483162,1.195671,0.198294,88.23,0.999608,100,1,1
119781,75581.0,-2.866364,2.346949,-4.053307,3.983359,-3.463186,-1.280953,-4.474764,1.216655,-2.309829,...,0.282030,-0.506901,-0.371741,0.615257,0.803163,124.53,0.999202,100,1,1


In [17]:
def explain_transaction(row):
    reasons = []

    if row["Amount"] > 200:
        reasons.append("High transaction amount")
    if row["score"] > 80:
        reasons.append("High risk score")
    return ", ".join(reasons)

top_risky["explanations"] = top_risky.apply(explain_transaction, axis=1)
top_risky

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V25,V26,V27,V28,Amount,prob,score,Class,pred,explanations
42009,40919.0,-2.740483,3.658095,-4.110636,5.340242,-2.666775,-0.092782,-4.388699,-0.280133,-2.821895,...,-0.403956,0.277895,0.830062,0.218690,112.33,0.999743,100,1,1,High risk score
15476,26863.0,-21.209120,12.652197,-23.553933,6.174078,-16.026658,-4.422195,-16.229444,14.116002,-3.922741,...,1.454857,-0.223214,1.550928,0.461460,99.99,0.999677,100,1,1,High risk score
229712,146022.0,0.908637,2.849024,-5.647343,6.009415,0.216656,-2.397014,-1.819308,0.338527,-2.819883,...,0.465058,0.210510,0.648705,0.360224,1.18,0.999692,100,1,1,High risk score
143333,85285.0,-7.030308,3.421991,-9.525072,5.270891,-4.024630,-2.865682,-6.989195,3.791551,-4.622730,...,0.353634,1.042458,1.359516,-0.272188,0.00,0.999748,100,1,1,High risk score
77348,57007.0,-1.271244,2.462675,-2.851395,2.324480,-1.372245,-0.948196,-3.065234,1.166927,-2.268771,...,0.224228,0.756335,0.632800,0.250187,0.01,0.999665,100,1,1,High risk score
6529,7891.0,-1.585505,3.261585,-4.137422,2.357096,-1.405043,-1.879437,-3.513687,1.515607,-1.207166,...,0.123644,0.321985,0.264028,0.132817,1.00,0.999317,100,1,1,High risk score
151519,95628.0,-17.518909,12.572118,-19.038538,11.190895,-13.554721,-0.411924,-23.189397,-5.301412,-8.630390,...,-0.720128,-0.232603,-3.021992,-0.478158,1.63,0.999587,100,1,1,High risk score
143336,85285.0,-6.713407,3.921104,-9.746678,5.148263,-5.151563,-2.099389,-5.937767,3.578780,-4.684952,...,0.394096,1.075295,1.649906,-0.394905,252.92,0.999754,100,1,1,"High transaction amount, High risk score"
42696,41203.0,-8.426814,6.241659,-9.946470,8.199614,-8.213093,-2.522046,-11.643028,5.339500,-7.051016,...,0.467594,0.483162,1.195671,0.198294,88.23,0.999608,100,1,1,High risk score
119781,75581.0,-2.866364,2.346949,-4.053307,3.983359,-3.463186,-1.280953,-4.474764,1.216655,-2.309829,...,-0.506901,-0.371741,0.615257,0.803163,124.53,0.999202,100,1,1,High risk score
